# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Research question:** Given a (client, content) page's first-half-of-month search activity, can
we identify — before the month ends — which pages are at risk of losing more than 20% of their
search impressions in the second half?

**Decision this supports:** which pages a content/SEO team should prioritize reviewing this
sprint, out of a fixed review budget, *before* a decline shows up in a monthly report rather than
after.

**Lane:** Freestyle — Growth / Recovery / Momentum Prediction, on the full FlyRank warehouse
release (`fact_content_daily_performance`, `month=2026-03`).


In [ ]:
print("Question: will a (client, content) pair's impressions drop >20% in the second half")
print("          of the month, predicted from first-half activity alone?")
print("Decision: which pages to prioritize for review, out of a fixed sprint budget.")
print("Lane: Freestyle -- Growth/Recovery/Momentum Prediction, warehouse month=2026-03")


## 2. Data

**Release:** FlyRank ML Internship warehouse (gated, Hugging Face, Parquet).

**Table used:** `fact_content_daily_performance`, partition `month=2026-03` — a mid-panel month,
never the sealed final-month `_sample` table. Grain: `report_date × client_hash_id ×
content_hash_id`, confirmed unique (0 duplicates) in `w03_data_contract.ipynb`.

**Date windows:** feature window Mar 1-15, label window Mar 16-31 — one month split in half so
the whole pipeline stays inside a single partition.

**Scale:** 9,841,378 raw daily rows, 55 active clients, 331,437 content items in the raw
partition; 151,981 (client, content) pairs with data in both halves after aggregation — the
actual modeling population.

**Excluded:** GA4-derived columns for this month (only 4.2% of rows have `ga4_data_available IS
TRUE`), any pre-computed FlyRank product flags, and — obviously — no client names, raw exports, or
private queries anywhere in this repo (public-safe throughout, per the capstone's public rule).


In [ ]:
data_summary = {
    "release": "FlyRank ML Internship warehouse (Hugging Face, gated)",
    "table": "fact_content_daily_performance, month=2026-03",
    "feature_window": "2026-03-01 to 2026-03-15",
    "label_window": "2026-03-16 to 2026-03-31",
    "raw_rows": 9_841_378,
    "clients": 55,
    "content_items": 331_437,
    "modeling_pairs": 151_981,
    "ga4_available_pct": 4.2,
}
import pandas as pd
pd.Series(data_summary)


## 3. Methodology

**Target:** `is_declining_next_half` = 1 if `impressions_second_half < 0.8 × impressions_first_half`,
else 0. An observed outcome from a genuinely future window, not a same-window bucket.

**Features (5, all Mar 1-15 only):** `impressions_first_half`, `clicks_first_half`,
`ctr_first_half`, `avg_position_first_half`, `active_days_first_half`. Verified disjoint from
label-side columns in `w03_feature_leakage_check.ipynb` and `w06_validation_audit.ipynb`.

**Baseline:** a transparent rule (`w04_baseline_score.ipynb`) — flag pages with sufficient volume
(≥50 impressions) and CTR below their own position tier's weighted average. No fitted weights.

**Validation design:** client-grouped 70/30 split (`w05_model.ipynb`) — no content item from a
train client appears in test, closing client-level leakage. Primary metric: Precision@10 on
held-out clients, matching the baseline's evaluation metric for a fair comparison.

**Leakage checks:** explicit trap test (train with vs without `impressions_second_half` as a
feature, `w03_feature_leakage_check.ipynb`), disjoint-set assertions in three separate notebooks,
and a schema scan for pre-computed FlyRank product flags.


In [ ]:
methodology = {
    "target": "is_declining_next_half (impressions_second_half < 0.8 x impressions_first_half)",
    "features": ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                 "avg_position_first_half", "active_days_first_half"],
    "baseline": "rule-based: volume >= 50 AND ctr_first_half < position-tier weighted CTR",
    "models_tried": ["Logistic Regression", "Random Forest"],
    "split": "client-grouped, 70% train clients / 30% test clients, random_state=42",
    "primary_metric": "Precision@10 on held-out test clients",
}
import pandas as pd
for k, v in methodology.items():
    print(f"{k}: {v}")


## 4. Results (vs baseline)

| Method | Evaluated on | Precision@10 |
|---|---|---|
| Rule baseline (`w04_baseline_score.ipynb`) | Whole month, not client-held-out | 50.0% |
| Base rate | Whole modeling population | 32.7% |
| Model (`w05_model.ipynb`) | Client-held-out test set | *pull from your own run — see w06 cell 4* |

The honest comparison point is the base rate (32.7%) — anything meaningfully above that on a
fixed-size top-K list represents real signal, not chance. The rule baseline already clears that
bar by a wide margin; the model's job (per `w06_validation_audit.ipynb`) is to clear it on
*held-out clients*, which is a fairer, harder bar than the baseline's whole-month number.


In [ ]:
results_table = {
    "Rule baseline (whole month)": {"precision_at_10": 0.500, "held_out_clients": False},
    "Base rate": {"precision_at_10": 0.327, "held_out_clients": False},
    # Fill this row in from your own w05_model.ipynb / w06_validation_audit.ipynb run:
    "Model (Logistic Regression)": {"precision_at_10": None, "held_out_clients": True},
}
import pandas as pd
pd.DataFrame(results_table).T


## 5. Limitations

- **Scaled-down window:** the Mar 1-15 / Mar 16-31 split is a stand-in for a production-style
  90-day/30-day comparison — it fits inside one warehouse partition on purpose, but low-volume
  pairs get noisier first-half estimates over only 15 days than a 90-day baseline would give.
- **Single month, single snapshot:** trained and validated on March 2026 only — not yet shown to
  generalize to other months or seasons.
- **Observational, not causal:** every relationship reported here is associative. No feature is
  claimed to *cause* a decline.
- **GA4 mostly unavailable this month:** only 4.2% of rows had GA4 data, so the model leans
  entirely on GSC signals (impressions, clicks, position) — engagement and scroll-depth signals
  that FlyRank's own Health Score uses are absent here.
- **Decision-support only:** the output is a ranked review list with reason codes, not an
  instruction to act, and never a claim about Google's ranking algorithm itself.


In [ ]:
limitations = [
    "15-day/15-day window is a scaled stand-in for a 90-day/30-day production comparison",
    "Single month (March 2026) -- not yet validated across seasons or other months",
    "Observational: no causal claims",
    "GA4 available for only 4.2% of this month's rows -- GSC-only feature set",
    "Decision-support output, not an automation or a claim about search-engine mechanics",
]
for l in limitations:
    print("-", l)


## 6. Ranked recommendations

From `w07_action_playbook.ipynb`'s exported queue (`work/outputs/action_playbook_queue.csv`):

1. **Review `high_volume_at_risk`-flagged pages first** — these carry the most impressions to
   lose if the flag is correct, so they're the highest expected-value use of review time.
2. **Treat `thin_coverage`-flagged pages as lower-confidence** — verify against daily-level data
   before acting; the model's read on these pairs is based on very few active days.
3. **Investigate `low_ctr_for_position` pages for metadata/snippet issues** — visibility (position)
   is fine, but clicks are underperforming what similar-position pages typically earn.
4. **Don't automate deletions, redirects, or de-indexing directly from this queue** — every flagged
   action needs a human check first (per `w07_action_playbook.ipynb`'s no-go list).


In [ ]:
# Load the exported queue from w07 and show the top of the actual recommendation list
import pandas as pd
queue = pd.read_csv("work/outputs/action_playbook_queue.csv")
print(f"Loaded {len(queue):,} ranked rows")
queue.head(10)


## 7. Artifacts the paper embeds

Charts and tables the deployed page should show: (1) the Results table above (baseline vs base
rate vs model), (2) a bar chart of reason-code counts from `action_playbook_metrics.json`, (3)
the signal-audit verdict summary from `w04_signal_audit.ipynb` (Test 1/2/3 + flag-linked test),
and (4) the paper's own correlation reference point (health score vs. position, r = -0.592) next
to this project's directional replication check.


In [ ]:
import json, matplotlib.pyplot as plt

with open("work/outputs/action_playbook_metrics.json") as f:
    metrics = json.load(f)

reason_counts = metrics["reason_code_counts"]
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(reason_counts.keys(), reason_counts.values())
ax.set_title("Flagged pages by reason code")
ax.set_ylabel("Count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("work/outputs/reason_code_chart.png", dpi=150)
plt.show()
print("Saved work/outputs/reason_code_chart.png for the deployed paper")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
